# Практикум: оценка численности населения

Данные о населении — одни из самых важных в городской аналитике. Но официальная статистика публикуется только для крупных единиц (районов, округов), а для отдельных домов или кварталов её нет. Поэтому численность населения на детальном уровне приходится **оценивать**.

Один из способов — использовать данные о многоквартирных домах (МКД). Если известна жилая площадь каждого дома и средняя обеспеченность жильём (м² на человека), можно оценить число жителей в каждом доме.

В этом практикуме мы применим инструменты модуля к реальной задаче:

- с помощью пространственного объединения определим округ каждого дома и рассчитаем обеспеченность жильём по округам (раздел [Пространственное объединение и агрегирование](geoprocessing_3.ipynb));
- оценим число жителей каждого дома;
- агрегируем оценки по регулярной сетке и построим карту плотности населения (раздел [Агрегирование точечных данных по регулярной сетке](geoprocessing_4.ipynb)).


#### Реестр многоквартирных домов

Реестр МКД, который ведётся в рамках реформы ЖКХ, содержит информацию о многоквартирных домах России:

- **идентификатор** и **адрес** дома;
- **характеристики**: год постройки, этажность, тип дома, общая, жилая и нежилая площадь помещений, состояние здания (аварийное или нет) и другое.

Реестр уже содержит координаты домов в полях `lat` и `lon`.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import pandas as pd
import geopandas as gpd

### 0.2 Подготовка данных

Используем файлы, с которыми мы уже работали в разделе [Пространственное объединение и агрегирование](geoprocessing_3.ipynb):

- **spb_admin.gpkg** — границы муниципальных округов Санкт-Петербурга (слой `okrug`), в поле `Popul` — численность населения;
- **spb_mkd_reforma.csv** — многоквартирные дома Санкт-Петербурга. Источник: [Фонд развития территорий](https://витрина.фрт.рф/opendata)


In [ ]:
okrugs = gpd.read_file("../data/spb_admin.gpkg", layer="okrug")

mkd = pd.read_csv("../data/spb_mkd_reforma.csv", index_col=0, low_memory=False)
mkd = mkd.dropna(subset=["lon", "lat"])

mkd_gdf = gpd.GeoDataFrame(
    mkd,
    geometry=gpd.points_from_xy(mkd["lon"], mkd["lat"]),
    crs=4326
)

## 1. Оценка численности жителей каждого дома

### 1.1. Пространственное объединение

Определим, в каком округе находится каждый дом. Оба слоя хранятся в EPSG:4326, поэтому перепроецирование не требуется. Сразу переименуем поля округа, чтобы с ними было удобнее работать:


In [ ]:
mkd_okrug = gpd.sjoin(
    mkd_gdf,
    okrugs[["NAME", "Popul", "geometry"]],
    how="inner",
    predicate="within"
).rename(columns={"NAME": "okrug_name", "Popul": "okrug_population"})

mkd_okrug = mkd_okrug.drop(columns="index_right")

mkd_okrug[["okrug_name", "okrug_population", "area_residential"]].head()

Мы использовали `how="inner"`: дома, которые не попали ни в один округ, нам не нужны.


### 1.2. Обеспеченность жильём по округам

Рассчитаем суммарную жилую площадь домов в каждом округе и разделим её на численность населения округа — получим среднюю обеспеченность жильём (м²/чел.):


In [ ]:
okrug_stat = (
    mkd_okrug
    .groupby("okrug_name")
    .agg(
        total_residential_area=("area_residential", "sum"),
        okrug_population=("okrug_population", "first"),
    )
    .reset_index()
)

okrug_stat["area_per_person"] = okrug_stat["total_residential_area"] / okrug_stat["okrug_population"]

okrug_stat.sort_values("area_per_person").head()

### 1.3. Проверка обеспеченности

Посмотрим на распределение обеспеченности по округам:


In [ ]:
okrug_stat["area_per_person"].describe()

Медианная обеспеченность по округам — около 23 м²/чел., но в части округов она близка к нулю. Причина в том, что реестр МКД **неполон**: в нём нет частных домов, у части домов не указана жилая площадь, а некоторые округа в реестре почти не представлены. Если использовать такую «обеспеченность», оценка населения домов получится абсурдной (тысячи или даже бесконечное число жителей в доме).

Оставим только округа с реалистичной обеспеченностью — от 10 до 60 м²/чел. Остальные округа исключим из оценки:


In [ ]:
valid = okrug_stat["area_per_person"].between(10, 60)

print(f"Округов с реалистичной обеспеченностью: {valid.sum()} из {len(okrug_stat)}")

okrug_stat = okrug_stat[valid]

### 1.4. Оценка численности жителей

Присоединим обеспеченность к каждому дому и оценим число его жителей: жилая площадь дома / обеспеченность жильём в округе. Объединение `how="inner"` оставит только дома в округах, прошедших проверку:


In [ ]:
mkd_okrug = mkd_okrug.merge(okrug_stat[["okrug_name", "area_per_person"]], on="okrug_name", how="inner")

mkd_okrug["estimated_population"] = mkd_okrug["area_residential"] / mkd_okrug["area_per_person"]

mkd_okrug[["okrug_name", "area_residential", "area_per_person", "estimated_population"]].head()

> **Ограничения оценки.**
>
> - Мы выводим обеспеченность жильём из **уже известной** численности населения округа, а затем распределяем её по домам пропорционально их жилой площади. Это не независимая оценка «с нуля», а **дезагрегация** известного значения: сумма оценок по домам округа равна его исходному населению.
> - Реестр МКД **неполон**: в нём нет частных домов, а часть многоквартирных домов может отсутствовать. Там, где домов в реестре мало, оценка менее надёжна.
>
> Поэтому мы исключили округа с нереалистичной обеспеченностью. На карте плотности они останутся пустыми: это честнее, чем показать заведомо неверную оценку.


## 2. Карта плотности населения

### 2.1. Регулярная сетка

Используем функцию построения сетки из раздела [Агрегирование точечных данных по регулярной сетке](geoprocessing_4.ipynb):


In [ ]:
from shapely.geometry import box


def create_regular_grid(data, cell_size):
    """Регулярная сетка квадратных ячеек по охвату data (в метрической СК)."""

    if data.crs is None:
        raise ValueError("У входных данных не задана система координат (CRS).")

    # Размер ячейки задаётся в метрах, поэтому нужна метрическая СК
    if data.crs.is_geographic:
        data = data.to_crs(data.estimate_utm_crs())

    minx, miny, maxx, maxy = data.total_bounds

    cells = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            cells.append(box(x, y, x + cell_size, y + cell_size))
            y += cell_size
        x += cell_size

    grid = gpd.GeoDataFrame({"geometry": cells}, crs=data.crs)
    grid["cell_id"] = range(len(grid))

    return grid

Перепроецируем данные в зону UTM и построим сетку с ячейками 1 × 1 км. Площадь ячейки — ровно 1 км², поэтому сумма жителей в ячейке сразу равна плотности населения (чел./км²):


In [ ]:
utm_crs = mkd_okrug.estimate_utm_crs()
mkd_utm = mkd_okrug.to_crs(utm_crs)

grid = create_regular_grid(mkd_utm, 1000)

### 2.2. Расчёт плотности населения

Определим ячейку каждого дома, просуммируем оценки населения по ячейкам и присоединим результат к сетке:


In [ ]:
mkd_in_grid = gpd.sjoin(mkd_utm, grid, how="inner", predicate="within")

pop_by_cell = (
    mkd_in_grid
    .groupby("cell_id")["estimated_population"]
    .sum()
    .reset_index(name="pop_sum")
)

pop_grid = grid.merge(pop_by_cell, on="cell_id", how="inner")
pop_grid["pop_density"] = pop_grid["pop_sum"] / (pop_grid.geometry.area / 1_000_000)

Мы использовали `how="inner"`, чтобы оставить только ячейки, в которых есть жилые дома.

### 2.3. Визуализация


In [ ]:
pop_grid.explore(
    column="pop_density",
    cmap="YlGnBu",
    scheme="NaturalBreaks",
    k=5,
    legend=True,
    legend_kwds={"caption": "Плотность населения, чел./км²"}
)

## Итог

В этом практикуме мы применили инструменты модуля к реальной задаче:

- с помощью пространственного объединения и агрегирования рассчитали обеспеченность жильём по округам;
- оценили численность жителей каждого многоквартирного дома;
- агрегировали оценки по регулярной сетке и построили карту плотности населения.

Такой подход пригодится, когда нужно оценить, сколько людей живёт вокруг объекта, например в зоне доступности школы или нового жилого комплекса.
